## TODO:

* copy all the stuff done in ccle for task 1
* do the task 2:

Look at the instructions on the mail 

(Buffa link that she looked at in class: https://portal.gdc.cancer.gov/analysis_page?app=Downloads), selected WGS, somatic structural variation, vcf, Manta, primary, solid tissue

In [1]:
import pandas as pd
import numpy as np
from sklearn.manifold import TSNE
import seaborn as sns
import umap
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import xgboost as xgb

/opt/anaconda3/envs/bio_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Import data

In [ ]:
rna = pd.read_csv("data_tcga/data_mrna_seq_fpkm.txt", sep="\t").T.reset_index()
rna.columns = rna.iloc[0]
rna = rna[1:] 
rna

,Hugo_Symbol,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,...,AP000230.1,RP11-80H18.4,RP13-297E16.4,LL0YNC03-29C1.1,RP13-297E16.5,BX649553.1,BX649553.3,BX649553.4,RN7SL355P,MIR3690
1,SP89389,65.515874,0.046881,63.315522,2.222857,2.565836,2.167233,1.265078,34.199996,4.974365,...,0.147707,0.24506,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,SP21193,93.507754,0.655002,23.608843,1.933762,1.861716,3.873639,3.268175,23.050032,4.019948,...,0.011216,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,SP13206,15.155525,0.0,42.136192,2.318698,2.87044,4.286835,9.066806,32.926751,3.909932,...,0.02706,0.179581,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,SP103623,9.75167,0.188735,14.507132,2.342167,0.454312,13.098299,26.108567,25.074303,0.682193,...,0.241574,1.479853,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,SP32742,16.04885,0.014542,45.302557,4.373694,4.125029,4.316576,16.914069,15.602162,8.076706,...,0.04009,0.15203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1206,SP16269,0.007476,0.0,23.290497,6.46653,6.384945,235.016264,0.013622,5.548667,13.125604,...,0.0,0.432225,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1207,SP122676,9.736614,0.062405,31.444337,1.072155,1.207673,5.166039,5.904768,61.047477,2.502007,...,0.0,0.733968,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1208,SP88776,29.283034,0.0,23.118048,3.699246,0.539174,0.450032,0.403973,25.097563,6.154079,...,0.751576,1.280037,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1209,SP64546,28.04515,0.026696,51.549932,1.989817,1.556905,1.276369,9.101945,15.649566,6.412556,...,0.087614,0.232576,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
mutation_type = pd.read_csv("data_tcga/mutations.txt", sep="\t")
mutation_type

,STUDY_ID,SAMPLE_ID,TP53
0,pancan_pcawg_2020,SP107436,WT
1,pancan_pcawg_2020,SP107435,WT
2,pancan_pcawg_2020,SP107407,WT
3,pancan_pcawg_2020,SP107406,WT
4,pancan_pcawg_2020,SP107405,WT
...,...,...,...
2678,pancan_pcawg_2020,SP8891,X187_splice
2679,pancan_pcawg_2020,SP88776,WT
2680,pancan_pcawg_2020,SP88757,WT
2681,pancan_pcawg_2020,SP88593,WT


In [ ]:
types_of_mutation = mutation_type["TP53"]
(Counter(types_of_mutation))

Counter({'WT': 1781,
         'R273H': 33,
         'R175H': 27,
         'R248Q': 25,
         'R282W': 23,
         'Y220C': 20,
         'R213*': 18,
         'R248W': 18,
         'X126_splice': 17,
         'R273C': 15,
         'G245S': 11,
         'R306*': 10,
         'R342*': 9,
         'K132N': 8,
         'R196*': 8,
         'H193R': 8,
         'X187_splice': 7,
         'X331_splice': 7,
         'H179R': 7,
         'X125_splice': 7,
         'X307_splice': 7,
         'M237I': 6,
         'I195T': 6,
         'X261_splice': 6,
         'S241F': 6,
         'Y163C': 6,
         'Y205C': 6,
         'Q331*': 6,
         'R337C': 6,
         'X225_splice': 5,
         'H214R': 5,
         'Q192*': 5,
         'V272M': 5,
         'S241C': 5,
         'C238F': 5,
         'G262V': 5,
         'R209Kfs*6': 5,
         'Q317*': 5,
         'V157F': 5,
         'X224_splice': 4,
         'E286K': 4,
         'W91*': 4,
         'X332_splice': 4,
         'A161T': 4,
        

## Task 2

### Look at Sean datasets

In [12]:
import gzip
import os

file_path = 'data_tcga/tcga(sean)/mrna_mutation_data/218cbd80-c866-48b9-90d6-d0a5210fd96d/579a3ebc-d38d-4029-a05c-d3e8ea2549d8.wxs.aliquot_ensemble_masked.maf.gz'

with gzip.open(file_path, 'rt') as f:
    lines = f.readlines()
    print(f"First 10 lines of {file_path}:")
    for line in lines:
        print(line.strip())

First 10 lines of data_tcga/tcga(sean)/mrna_mutation_data/218cbd80-c866-48b9-90d6-d0a5210fd96d/579a3ebc-d38d-4029-a05c-d3e8ea2549d8.wxs.aliquot_ensemble_masked.maf.gz:
#version gdc-1.0.0
#annotation.spec gdc-2.0.0-aliquot-merged-masked
#contigs chr1,chr2,chr3,chr4,chr5,chr6,chr7,chr8,chr9,chr10,chr11,chr12,chr13,chr14,chr15,chr16,chr17,chr18,chr19,chr20,chr21,chr22,chrX,chrY,chrM
#sort.order BarcodesAndCoordinate
#filedate 20220519
#normal.aliquot 0ffd9ad1-6643-42ff-af29-13389bfb1364
#tumor.aliquot 49fb101d-f242-4abf-bf85-6a8c3ced711d
Hugo_Symbol	Entrez_Gene_Id	Center	NCBI_Build	Chromosome	Start_Position	End_Position	Strand	Variant_Classification	Variant_Type	Reference_Allele	Tumor_Seq_Allele1	Tumor_Seq_Allele2	dbSNP_RS	dbSNP_Val_Status	Tumor_Sample_Barcode	Matched_Norm_Sample_Barcode	Match_Norm_Seq_Allele1	Match_Norm_Seq_Allele2	Tumor_Validation_Allele1	Tumor_Validation_Allele2	Match_Norm_Validation_Allele1	Match_Norm_Validation_Allele2	Verification_Status	Validation_Status	Mutation_S

In [11]:
len(lines)

42

In [ ]:
# make lines into pandas dataframe
import pandas as pd
import gzip
import os



['PIK3CA\t5290\tWUGSC\tGRCh38\tchr3\t179199073\t179199073\t+\tMissense_Mutation\tSNP\tT\tT\tC\trs1560137208\t\tTCGA-XX-A899-01A-11D-A36J-09\tTCGA-XX-A899-10A-01D-A36M-09\t\t\t\t\t\t\t\t\tSomatic\t\t\t\t\t\t\t49fb101d-f242-4abf-bf85-6a8c3ced711d\t0ffd9ad1-6643-42ff-af29-13389bfb1364\tc.248T>C\tp.Phe83Ser\tp.F83S\tENST00000263967\t2/21\t131\t103\t28\t134\t\t\tPIK3CA,missense_variant,p.F83S,ENST00000263967,NM_006218.4,c.248T>C,MODERATE,YES,deleterious(0.02),benign(0.06),1;PIK3CA,missense_variant,p.F83S,ENST00000643187,,c.248T>C,MODERATE,,deleterious(0.01),benign(0.283),1;PIK3CA,missense_variant,p.F83S,ENST00000468036,,c.248T>C,MODERATE,,tolerated(0.33),benign(0.06),1;PIK3CA,downstream_gene_variant,,ENST00000477735,,,MODIFIER,,,,1;PIK3CA,missense_variant,p.F83S,ENST00000675786,,c.248T>C,MODERATE,,deleterious(0),possibly_damaging(0.47),1;PIK3CA,non_coding_transcript_exon_variant,,ENST00000675467,,n.3055T>C,MODIFIER,,,,1;PIK3CA,upstream_gene_variant,,ENST00000674534,,,MODIFIER,,,,1\tC\tENSG0

In [4]:
import pandas as pd

def analyze_tsv_file(file_path):
    df = pd.read_csv(file_path, sep='\t', nrows=10)
    print(f"First 10 rows of {file_path}:")
    print(df)

# Example usage
tsv_file_path = 'data_tcga/tcga(sean)/mrna_expression_data/515f8866-8744-44d8-9e4b-0e734d0c201b/c5d73777-b7f3-4107-9efc-afde92f70469.rna_seq.augmented_star_gene_counts.tsv'
analyze_tsv_file(tsv_file_path)

First 10 rows of data_tcga/tcga(sean)/mrna_expression_data/515f8866-8744-44d8-9e4b-0e734d0c201b/c5d73777-b7f3-4107-9efc-afde92f70469.rna_seq.augmented_star_gene_counts.tsv:
                                                                                                                     # gene-model: GENCODE v36
gene_id            gene_name gene_type      unstranded stranded_first stranded_second tpm_unstranded fpkm_unstranded        fpkm_uq_unstranded
N_unmapped         NaN       NaN            2476851    2476851        2476851         NaN            NaN                                   NaN
N_multimapping     NaN       NaN            4572009    4572009        4572009         NaN            NaN                                   NaN
N_noFeature        NaN       NaN            2235792    24766058       24872173        NaN            NaN                                   NaN
N_ambiguous        NaN       NaN            5388767    1252536        1243009         NaN            NaN        